1. What is a Retriever?
A retriever is a LangChain component designed to fetch relevant documents from a data source based on a user's query

How it Works: It acts as a search function—taking a user query as input, searching the underlying data source, and outputting the most relevant results as a list of LangChain Document objects

Runnables: Crucially, all LangChain retrievers are "Runnables"
This means they possess the standard .invoke() method and can be seamlessly plugged into chains alongside LLMs and Prompts, maximizing the flexibility of your RAG pipeline

2. Types of Retrievers
LangChain offers dozens of retrievers tailored for specific use cases, which generally fall into two broad categories
:
By Data Source: Retrievers built to interact with specific external sources (e.g., Wikipedia, Arxiv, or Vector Stores)
.
By Search Strategy: Retrievers that implement specialized search algorithms or mechanisms to improve the quality of the retrieved results
.


In [ ]:
# A. Wikipedia Retriever (Data Source Based)
# This retriever connects to the Wikipedia API
# . Instead of using vector-based semantic search, it utilizes key-word matching to find and return the most relevant Wikipedia articles for a given query

from langchain_community.retrievers import WikipediaRetriever

# Initialize the Retriever
# You can specify the number of results (top_k_results) and the language
retriever = WikipediaRetriever(top_k_results=2, lang="en")

# Execute the search
query = "The geopolitical history of India and Pakistan"
docs = retriever.invoke(query)

# View the results
for doc in docs:
    print(doc.page_content)

In [ ]:
# B. Vector Store Retriever (Data Source Based)
# This is the most common retriever
# . It connects to a Vector Store (like Chroma or FAISS) and uses vector embeddings to perform a semantic similarity search

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Assuming you already have 'docs' prepared
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())

# Convert the Vector Store into a Retriever
# k=2 tells it to return the top 2 most relevant documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

results = retriever.invoke("What is Chroma used for?")


In [ ]:
# C. Maximum Marginal Relevance (MMR) Retriever (Strategy Based)
# The Problem: Standard similarity search often returns redundant information
# . If a user asks about the effects of climate change, the top three results might all talk about melting glaciers, completely ignoring other effects like wildfires or deforestation
# . The Solution: MMR solves this by balancing relevance with diversity
# . It selects the most relevant document first, but for the next selection, it purposely finds a document that is both relevant to the query AND dissimilar to the previously selected documents

# Create an MMR Retriever from a vector store
retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 3, 
        "lambda_mult": 0.5  # 1 = pure similarity (no diversity), 0 = maximum diversity
    }
)

results = retriever.invoke("What are the adverse effects of climate change?")

In [ ]:
# . Multi-Query Retriever (Strategy Based)
# The Problem: Users frequently ask ambiguous or poorly phrased questions (e.g., "How can I stay healthy?")
# . A single semantic search on a vague query often returns poor or mixed results
# . The Solution: This retriever passes the user's vague query to an LLM first
# . The LLM generates several specific, related queries from different angles (e.g., "What foods should I eat?", "How often should I exercise?")
# . The retriever then performs a semantic search for each of these generated queries, merges all the resulting documents, removes any duplicates, and returns a highly comprehensive set of results

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# Base retriever for the actual searching
base_retriever = vectorstore.as_retriever(search_type="similarity")

# Initialize Multi-Query Retriever with an LLM
advanced_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=ChatOpenAI(temperature=0)
)
results = advanced_retriever.invoke("How to improve energy levels and maintain balance")
